# Packages

In [2]:
import pandas as pd
import numpy as np
import os
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
dir = os.getenv('DIR')
os.getcwd()
os.chdir(dir)
path = os.getcwd()

In [117]:
df = pd.read_csv(os.path.join(path,'data','final','Dataset.csv'))

In [119]:
df['Key'] = df['Key'].apply(lambda x: str(x))

In [120]:
df

,Key,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,DR,...,Blk,PF,FG%,3P%,FT%,Possessions,NumOT,GamesPlayed,WinRate,RollingScore
0,1101,0.924,0.701,0.878,0.436,0.648,0.596,0.664,0.491,0.673,...,0.282,0.672,0.079,0.065,0.117,0.924,0.016,1.274,0.465,0.926
1,1102,0.905,0.682,0.852,0.466,0.675,0.555,0.631,0.444,0.672,...,0.275,0.633,0.081,0.066,0.114,0.901,0.009,1.440,0.401,0.915
2,1103,0.935,0.711,0.881,0.483,0.692,0.589,0.662,0.530,0.696,...,0.324,0.645,0.081,0.067,0.115,0.918,0.016,1.437,0.660,0.965
3,1104,0.939,0.715,0.885,0.463,0.678,0.606,0.678,0.549,0.710,...,0.374,0.640,0.081,0.064,0.116,0.923,0.015,1.455,0.614,0.985
4,1105,0.910,0.686,0.880,0.408,0.642,0.594,0.678,0.541,0.690,...,0.330,0.655,0.074,0.058,0.111,0.925,0.013,1.409,0.324,0.935
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,3477,0.907,0.691,0.894,0.419,0.653,0.557,0.627,0.491,0.699,...,0.351,0.616,0.071,0.058,0.118,0.937,0.008,1.031,0.369,0.896
756,3478,0.877,0.655,0.868,0.414,0.650,0.540,0.597,0.490,0.676,...,0.322,0.591,0.068,0.058,0.122,0.913,0.012,0.985,0.381,0.883
757,3479,0.906,0.678,0.874,0.481,0.707,0.560,0.622,0.430,0.653,...,0.259,0.643,0.073,0.062,0.121,0.931,0.017,0.863,0.366,0.933
758,3480,0.920,0.698,0.886,0.451,0.688,0.582,0.659,0.515,0.692,...,0.369,0.630,0.075,0.058,0.114,0.935,0.030,0.875,0.473,0.929


In [20]:
batch_size = 64
teams = 2
d_input = df.shape[1]-1


# Model

In [48]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Data

## Components

## Training only the Encoder first

In [121]:
df

,Key,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,OR,DR,...,Blk,PF,FG%,3P%,FT%,Possessions,NumOT,GamesPlayed,WinRate,RollingScore
0,1101,0.924,0.701,0.878,0.436,0.648,0.596,0.664,0.491,0.673,...,0.282,0.672,0.079,0.065,0.117,0.924,0.016,1.274,0.465,0.926
1,1102,0.905,0.682,0.852,0.466,0.675,0.555,0.631,0.444,0.672,...,0.275,0.633,0.081,0.066,0.114,0.901,0.009,1.440,0.401,0.915
2,1103,0.935,0.711,0.881,0.483,0.692,0.589,0.662,0.530,0.696,...,0.324,0.645,0.081,0.067,0.115,0.918,0.016,1.437,0.660,0.965
3,1104,0.939,0.715,0.885,0.463,0.678,0.606,0.678,0.549,0.710,...,0.374,0.640,0.081,0.064,0.116,0.923,0.015,1.455,0.614,0.985
4,1105,0.910,0.686,0.880,0.408,0.642,0.594,0.678,0.541,0.690,...,0.330,0.655,0.074,0.058,0.111,0.925,0.013,1.409,0.324,0.935
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
755,3477,0.907,0.691,0.894,0.419,0.653,0.557,0.627,0.491,0.699,...,0.351,0.616,0.071,0.058,0.118,0.937,0.008,1.031,0.369,0.896
756,3478,0.877,0.655,0.868,0.414,0.650,0.540,0.597,0.490,0.676,...,0.322,0.591,0.068,0.058,0.122,0.913,0.012,0.985,0.381,0.883
757,3479,0.906,0.678,0.874,0.481,0.707,0.560,0.622,0.430,0.653,...,0.259,0.643,0.073,0.062,0.121,0.931,0.017,0.863,0.366,0.933
758,3480,0.920,0.698,0.886,0.451,0.688,0.582,0.659,0.515,0.692,...,0.369,0.630,0.075,0.058,0.114,0.935,0.030,0.875,0.473,0.929


In [122]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 760 entries, 0 to 759
Data columns (total 23 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Key           760 non-null    object 
 1   Score         760 non-null    float64
 2   FGM           760 non-null    float64
 3   FGA           760 non-null    float64
 4   FGM3          760 non-null    float64
 5   FGA3          760 non-null    float64
 6   FTM           760 non-null    float64
 7   FTA           760 non-null    float64
 8   OR            760 non-null    float64
 9   DR            760 non-null    float64
 10  Ast           760 non-null    float64
 11  TO            760 non-null    float64
 12  Stl           760 non-null    float64
 13  Blk           760 non-null    float64
 14  PF            760 non-null    float64
 15  FG%           760 non-null    float64
 16  3P%           760 non-null    float64
 17  FT%           760 non-null    float64
 18  Possessions   760 non-null    

In [123]:
df.set_index('Key',inplace=True)

In [50]:
keys = df.index.tolist()   # Storing the keys
# keys

In [124]:
df.index

Index(['1101', '1102', '1103', '1104', '1105', '1106', '1107', '1108', '1109',
       '1110',
       ...
       '3472', '3473', '3474', '3475', '3476', '3477', '3478', '3479', '3480',
       '3481'],
      dtype='object', name='Key', length=760)

In [18]:
class TeamDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data.values, dtype=torch.float32)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [19]:
class TeamEncoder(nn.Module):
    def __init__(self, input_dim, d_model=128, n_heads=4, n_layers=2):
        super().__init__()

        # Project stats → embedding
        self.input_proj = nn.Linear(input_dim, d_model)

        # CLS token
        self.cls = nn.Parameter(torch.randn(1, 1, d_model))

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=256,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Decoder (for reconstruction)
        self.decoder = nn.Sequential(
            nn.Linear(d_model, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):
        # x: (B, input_dim)

        B = x.size(0)

        x = self.input_proj(x)              # (B, d_model)
        x = x.unsqueeze(1)                 # (B, 1, d_model)

        cls = self.cls.expand(B, 1, -1)    # (B, 1, d_model)

        x = torch.cat([cls, x], dim=1)     # (B, 2, d_model)

        x = self.encoder(x)                # (B, 2, d_model)

        cls_out = x[:, 0, :]               # (B, d_model)

        recon = self.decoder(cls_out)      # (B, input_dim)

        return recon, cls_out

In [20]:
def train_encoder(model, dataloader, epochs=20, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        total_loss = 0

        for batch in dataloader:
            x = batch.to(device)  # (B, input_dim)

            recon, _ = model(x)

            loss = criterion(recon, x)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [125]:
teams = TeamDataset(df)
teamsdataloader = torch.utils.data.DataLoader(teams, batch_size=64, shuffle=False)

encoder = TeamEncoder(input_dim=df.shape[1])

train_encoder(encoder, teamsdataloader)

Epoch 1, Loss: 1.7476
Epoch 2, Loss: 0.2425
Epoch 3, Loss: 0.1869
Epoch 4, Loss: 0.1709
Epoch 5, Loss: 0.1133
Epoch 6, Loss: 0.0506
Epoch 7, Loss: 0.0370
Epoch 8, Loss: 0.0356
Epoch 9, Loss: 0.0330
Epoch 10, Loss: 0.0329
Epoch 11, Loss: 0.0321
Epoch 12, Loss: 0.0312
Epoch 13, Loss: 0.0303
Epoch 14, Loss: 0.0289
Epoch 15, Loss: 0.0287
Epoch 16, Loss: 0.0271
Epoch 17, Loss: 0.0290
Epoch 18, Loss: 0.0266
Epoch 19, Loss: 0.0253
Epoch 20, Loss: 0.0267


In [22]:
class MatchPredictor(nn.Module):
    def __init__(self, encoder, d_model):
        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, teamA, teamB):
        _, zA = self.encoder(teamA)   # (B, d)
        _, zB = self.encoder(teamB)

        x = torch.cat([
            zA - zB,
            zA * zB,
            zA,
            zB
        ], dim=-1)

        return torch.sigmoid(self.classifier(x))

## Match pairs making

In [132]:
matchups = pd.read_csv(os.path.join(path,'data','final','Matchups.csv'))


In [133]:
matchups

,Unnamed: 0,Season,DayNum,TeamID,Score,OppTeamID,OppScore,Loc,NumOT,FGM,...,LastDayNum,CoachName,ConfAbbrev,Tournament,FG%,3P%,FT%,Possessions,Win,Key
0,0,2003,10,1104,68,1328,62,N,0,27,...,154.0,mark_gottfried,sec,Regular,0.465517,0.214270,0.611077,74.92044,1,1104-1328
1,1,2003,10,1272,70,1393,63,N,0,26,...,154.0,john_calipari,cusa,Regular,0.419355,0.399980,0.526288,68.36044,1,1272-1393
2,2,2003,10,1328,62,1104,68,N,0,22,...,154.0,kelvin_sampson,big_twelve,Regular,0.415094,0.199980,0.727240,70.68044,0,1328-1104
3,3,2003,10,1393,63,1272,70,N,0,24,...,154.0,jim_boeheim,big_east,Regular,0.358209,0.249990,0.449978,67.80044,0,1393-1272
4,4,2003,11,1186,55,1458,81,A,0,20,...,154.0,ray_giacoletti,big_sky,Regular,0.434783,0.272702,0.705841,66.48044,0,1186-1458
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428887,428887,2026,118,3449,70,3332,69,A,0,27,...,NaN,NaN,big_ten,Regular,0.457627,0.333321,0.583285,61.28044,1,3449-3332
428888,428888,2026,118,3452,118,3153,60,H,0,42,...,NaN,NaN,big_twelve,Regular,0.583333,0.499983,0.633312,82.20044,1,3452-3153
428889,428889,2026,118,3456,60,3423,41,A,0,22,...,NaN,NaN,caa,Regular,0.360656,0.333315,0.714235,64.16044,1,3456-3423
428890,428890,2026,118,3458,52,3234,81,H,0,19,...,NaN,NaN,big_ten,Regular,0.301587,0.241371,0.874891,61.52044,0,3458-3234


In [135]:
matchups.drop(columns=['Unnamed: 0'],inplace=True)

In [136]:
target = matchups[['Key','Win']]

In [137]:
target

,Key,Win
0,1104-1328,1
1,1272-1393,1
2,1328-1104,0
3,1393-1272,0
4,1186-1458,0
...,...,...
428887,3449-3332,1
428888,3452-3153,1
428889,3456-3423,1
428890,3458-3234,0


In [138]:
target['TeamAID'] = target['Key'].apply(lambda x: x.split('-')[0])
target['TeamBID'] = target['Key'].apply(lambda x: x.split('-')[1])

In [139]:
target['TeamAID'] = target['TeamAID'].apply(lambda x: str(x))
target['TeamBID'] = target['TeamBID'].apply(lambda x: str(x))

In [140]:
target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 428892 entries, 0 to 428891
Data columns (total 4 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   Key      428892 non-null  object
 1   Win      428892 non-null  int64 
 2   TeamAID  428892 non-null  object
 3   TeamBID  428892 non-null  object
dtypes: int64(1), object(3)
memory usage: 13.1+ MB


In [141]:
target.nunique()

Key        83094
Win            2
TeamAID      739
TeamBID      739
dtype: int64

In [27]:
target.index

RangeIndex(start=0, stop=428892, step=1)

In [142]:
encoder.eval()

team_embeddings = {}

with torch.no_grad():
    for team_id, stats in df.iterrows():
        stats = torch.tensor(stats.values, dtype=torch.float32).unsqueeze(0).to(device)

        _, emb = encoder(stats)   # (1, d)

        team_embeddings[team_id] = emb.squeeze(0).cpu()

In [143]:
team_ids = sorted(team_embeddings.keys())

team_embeddings = torch.stack([
    team_embeddings[tid] for tid in team_ids
])

In [59]:
pd.Series(team_ids).to_csv(os.path.join(path,'data','processed','Team_IDs.csv'),index=False)

In [144]:
team_embeddings.shape

torch.Size([760, 128])

In [145]:
target = target[['TeamAID','TeamBID','Win']]

In [146]:
target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 428892 entries, 0 to 428891
Data columns (total 3 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   TeamAID  428892 non-null  object
 1   TeamBID  428892 non-null  object
 2   Win      428892 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 9.8+ MB


In [147]:
target

,TeamAID,TeamBID,Win
0,1104,1328,1
1,1272,1393,1
2,1328,1104,0
3,1393,1272,0
4,1186,1458,0
...,...,...,...
428887,3449,3332,1
428888,3452,3153,1
428889,3456,3423,1
428890,3458,3234,0


In [97]:
idx = pd.read_csv(os.path.join(path,'data','processed','Team_IDs.csv'))

In [108]:
idx = idx.apply(lambda x: str(x['TeamID']),axis=1)

In [148]:
idx.values.ravel().tolist().index('1272')

171

In [149]:
team_embeddings[0]

tensor([ 2.1203e+00, -1.8483e-01,  9.2156e-01,  1.0422e+00, -5.3552e-02,
        -2.6634e-01, -1.8565e-01,  1.4524e-01, -2.6058e-02,  4.9495e-01,
        -2.3430e-01,  1.0294e+00,  1.4901e+00,  4.4383e-01, -5.2197e-01,
        -4.0267e-01, -9.4225e-01,  3.9029e-01, -5.5989e-01, -4.2409e-01,
         1.8003e+00, -3.1476e-01,  1.3862e+00, -1.6858e-01,  1.3824e+00,
         1.6668e+00, -9.1296e-01, -3.0560e-01,  1.2569e+00,  3.9873e-01,
         8.7135e-01, -1.4814e+00, -7.8213e-01,  4.0225e-01, -6.1636e-01,
         3.4880e-01,  5.2029e-01, -8.2192e-02, -6.3061e-01, -8.4152e-01,
        -2.1421e+00,  2.2898e-01, -5.7646e-02,  1.0296e-01,  1.9588e+00,
        -5.8749e-01, -1.1686e+00, -2.6485e-01,  5.2004e-01,  1.3727e+00,
        -3.4000e-01,  4.9180e-02, -9.9324e-01,  1.3307e+00, -7.1428e-03,
        -3.3966e-01,  1.3476e+00, -4.7773e-01, -9.7133e-02,  2.7258e+00,
         4.9743e-01, -5.7308e-02, -6.1745e-01, -1.2764e+00,  2.1627e+00,
        -1.3520e+00, -5.5669e-01, -2.4362e-01,  8.6

In [65]:
class MatchDatasetEmb(Dataset):
    def __init__(self, match_data, embedding_matrix):
        """
        match_data: list of (teamA_id, teamB_id, label)
        embedding_matrix: tensor (num_teams, d_model)
        """
        self.matches = match_data
        self.emb = embedding_matrix

    def __len__(self):
        return len(self.matches)

    def __getitem__(self, idx):
        row = self.matches.iloc[idx]
        teamA_id = row["TeamAID"]
        teamB_id = row["TeamBID"]
        label = row["Win"]

        # match team embeddings based on their keys
        teamA = team_ids.index(teamA_id)
        teamB = team_ids.index(teamB_id)

        zA = self.emb[teamA]
        zB = self.emb[teamB]

        return zA, zB, torch.tensor(label, dtype=torch.float32)

In [38]:
class MatchPredictorEmb(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.classifier = nn.Sequential(
            nn.Linear(d_model * 4, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, zA, zB):
        x = torch.cat([
            zA - zB,
            zA * zB,
            zA,
            zB
        ], dim=-1)

        return self.classifier(x)  # logits

In [39]:
def train_classifier(model, dataloader, epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for zA, zB, y in dataloader:
            zA = zA.to(device)
            zB = zB.to(device)
            y = y.to(device)

            logits = model(zA, zB).squeeze()
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

In [40]:
from torch.utils.data import DataLoader

In [151]:
matchups = MatchDatasetEmb(target, team_embeddings)

matchupsdataloader = DataLoader(
    matchups,
    batch_size=64, 
    shuffle=True
)

In [152]:
d_model = team_embeddings.shape[1]

classifier = MatchPredictorEmb(d_model)

train_classifier(classifier, matchupsdataloader, epochs=10, lr=1e-3)

Epoch 1/10 | Loss: 0.6155
Epoch 2/10 | Loss: 0.6117
Epoch 3/10 | Loss: 0.6111
Epoch 4/10 | Loss: 0.6103
Epoch 5/10 | Loss: 0.6102
Epoch 6/10 | Loss: 0.6097
Epoch 7/10 | Loss: 0.6097
Epoch 8/10 | Loss: 0.6095
Epoch 9/10 | Loss: 0.6093
Epoch 10/10 | Loss: 0.6092


In [153]:
def predict(model, teamA_id, teamB_id, embedding_matrix):
    device = next(model.parameters()).device

    teamA = team_ids.index(teamA_id)
    teamB = team_ids.index(teamB_id)

    
    zA = embedding_matrix[teamA].unsqueeze(0).to(device)
    zB = embedding_matrix[teamB].unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(zA, zB)
        prob = torch.sigmoid(logits)

    return prob.item()

In [155]:
p = predict(classifier, teamA_id='1104', teamB_id='1328', embedding_matrix=team_embeddings)
print("P(A wins):", p)

P(A wins): 0.5459277033805847


In [158]:
file = pd.read_csv(os.path.join(path,'data','raw','SampleSubmissionStage2.csv'))

In [159]:
file

,ID,Pred
0,2026_1101_1102,0.5
1,2026_1101_1103,0.5
2,2026_1101_1104,0.5
3,2026_1101_1105,0.5
4,2026_1101_1106,0.5
...,...,...
132128,2026_3478_3480,0.5
132129,2026_3478_3481,0.5
132130,2026_3479_3480,0.5
132131,2026_3479_3481,0.5


In [166]:
file.iloc[[0],[0]].values[0][0].split('_')[1]

'1101'

In [167]:
ids = file['ID'].apply(lambda x: x.split('_')).to_list()

preds = []
for i in range(len(file)):
    season = file.iloc[[i],[0]].values[0][0].split('_')[0]
    teamA_id = file.iloc[[i],[0]].values[0][0].split('_')[1]
    teamB_id = file.iloc[[i],[0]].values[0][0].split('_')[2]
    
    key = season + '_' + teamA_id + '_' + teamB_id

    predictions = predict(classifier,teamA_id,teamB_id,embedding_matrix=team_embeddings)

    preds.append([key,predictions])

preds = pd.DataFrame(preds,columns=['ID','Pred'])

preds.to_csv(os.path.join(path,'data','final','SampleSubmissionStage2.csv'))

In [168]:
preds.to_csv(os.path.join(path,'data','final','SampleSubmissionStage2.csv'),index=False)